# 概述

中间件提供了一种更严格地控制代理内部运行机制的方法。中间件的用途包括：

- 通过日志记录、分析和调试来跟踪代理行为。
- 转换提示、[工具选择](https://docs.langchain.com/oss/python/langchain/middleware/built-in#llm-tool-selector)和输出格式。
- 添加[重试](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-retry)、[回退](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-fallback)和提前终止逻辑。
- 应用[速率限制](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-call-limit)、防护措施和[个人身份信息检测](https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection)。

通过向以下参数传递中间件来添加它们[`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent)：

```python
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware

agent = create_agent(
    model="gpt-4.1",
    tools=[...],
    middleware=[
        SummarizationMiddleware(...),
        HumanInTheLoopMiddleware(...)
    ],
)
```

#  The agent loop 代理循环

核心代理循环包括调用模型，让模型选择要执行的工具，然后在不再需要调用任何工具时结束：

![image-20260411213427746](E:\code\LangChainDemo\official_documentation\images\middleware_overview.png)

中间件在每个步骤之前和之后都会暴露钩子（Hook）：

![image-20260411213608801](E:\code\LangChainDemo\official_documentation\images\middleware_hook.png)

## 🔥 一句话总览

👉 本质就是一条链：

```
请求 → Agent前处理 → 模型调用（可插钩子）→ 工具调用（可插钩子）→ 收尾 → 返回
```

------

## 🧠 按执行顺序逐步拆解（重点！）

### 1️⃣ request（入口）

用户请求进来，比如：

```
agent.invoke("帮我查天气")
```

------

### 2️⃣ before_agent

👉 **Agent级别的前置处理**

你可以在这里做：

- 改写输入
- 注入系统 prompt
- 做鉴权 / logging

✔️ 这是“最外层”的 hook

------

### 3️⃣ before_model

👉 在真正调用 LLM 之前

可以做：

- prompt 最终拼装
- 加 memory
- 加工具描述

------

## ⚠️ 关键分叉来了（核心理解点）

### 🧩 分成两条路：

```
           before_model
           /        \
   工具路径          模型路径
```

------

### 4️⃣ wrap_model_call（模型调用包装）

👉 这是包住 LLM 的一层 middleware

你可以在这里：

- 打日志
- 做 retry
- 统计 token
- 替换模型

本质：

```
def wrap_model_call(model):
    # before
    result = model(...)
    # after
    return result
```

------

### 5️⃣ after_model

👉 模型返回之后

可能发生两种情况：

#### ✅ 情况1：模型直接回答

→ 直接往下走

#### ✅ 情况2：模型决定调用工具

→ 进入左边那条线 👇

------

### 6️⃣ wrap_tool_call（工具调用包装）

👉 如果模型输出了 tool call，就会走这里

你可以：

- 控制工具执行
- mock 工具
- 加缓存
- 做限流

本质：

```
def wrap_tool_call(tool):
    result = tool(...)
    return result
```

------

#### 🔁 注意这个循环（图里虚线）

工具执行完 → 会回到：

```
after_model → before_model → 再调模型
```

👉 也就是经典 Agent loop：

```
LLM → Tool → LLM → Tool → ... → Final Answer
```

------

### 7️⃣ after_agent

👉 整个 Agent 完成之后

可以做：

- 统一格式输出
- logging
- 监控

------

### 8️⃣ result（最终返回）

返回给用户：

```
"今天北京晴，25度"
```

------

## 🧠 “工程视角总结”（非常关键）

这张图其实在表达 3 层能力：

------

### 🥇 1. 生命周期 Hook（before / after）

- before_agent
- before_model
- after_model
- after_agent

👉 控制“什么时候做什么”

------

### 🥈 2. Middleware（wrap_*）

- wrap_model_call
- wrap_tool_call

👉 控制“调用本身”

------

### 🥉 3. Agent Loop（核心机制）

```
模型 → 判断 → 调工具 → 再问模型
```

👉 这才是 Agent 和普通 LLM 的本质区别